# Airbnb: анализ первого бронирования новых пользователей

**Источник:** [Airbnb New User Bookings на Kaggle](https://www.kaggle.com/competitions/airbnb-recruiting-new-user-bookings/data). Для повторного запуска требуется доступ к исходным данным согласно правилам соревнования; запускайте ячейки **сверху вниз**. Анализируются пользователи, зарегистрированные с января по июнь 2014 года. Имена исходных полей и действий оставлены без перевода, чтобы код работал с CSV.


In [1]:
from pathlib import Path
from zipfile import ZipFile
from shutil import copyfileobj, disk_usage
from collections import Counter
import pandas as pd

INPUT = Path('/kaggle/input')
TEMP = Path('/kaggle/temp')
OUT = Path('/kaggle/working')
TEMP.mkdir(parents=True, exist_ok=True)

def find_file(name):
    files = list(INPUT.rglob(name))
    if len(files) != 1:
        raise FileNotFoundError(f'Ожидался один файл {name}; найдено: {len(files)}')
    return files[0]

users_zip = find_file('train_users_2.csv.zip')
sessions_zip = find_file('sessions.csv.zip')
users = pd.read_csv(users_zip, compression='zip')
assert users['id'].notna().all() and users['id'].is_unique, 'Проверьте уникальность id'
users['registration'] = pd.to_datetime(users['date_account_created'], errors='coerce')
users['booking_date'] = pd.to_datetime(users['date_first_booking'], errors='coerce')
users['booked'] = users['country_destination'].ne('NDF')
assert (users['booked'] == users['booking_date'].notna()).all(), 'Не совпадают определения бронирования'
cohort = users[users['registration'].between('2014-01-01', '2014-06-30')].copy()
cohort['month'] = cohort['registration'].dt.to_period('M').astype(str)
print('Пользователей всего:', len(users))
print('Пользователей в когорте:', len(cohort))
print('С зафиксированным бронированием:', int(cohort['booked'].sum()))
print('Бронирований до регистрации:', int((users['booking_date'] < users['registration']).sum()))
print('Последняя дата бронирования:', users['booking_date'].max())


Пользователей всего: 213451
Пользователей в когорте: 76466
С зафиксированным бронированием: 29239
Бронирований до регистрации: 29
Последняя дата бронирования: 2015-06-29 00:00:00


## Определение метрик

Доля пользователей с зафиксированным бронированием = число пользователей с `country_destination != 'NDF'` / число пользователей в выборке. Это **не** конверсия за фиксированные 30 дней. Без даты каждого действия нельзя утверждать, что оно произошло до первого бронирования, или строить последовательную воронку.


In [2]:
sessions_csv = TEMP / 'sessions.csv'
with ZipFile(sessions_zip) as archive:
    members = [m for m in archive.infolist() if m.filename.endswith('sessions.csv')]
    if len(members) != 1:
        raise ValueError('В архиве должен быть ровно один sessions.csv')
    member = members[0]
    if not sessions_csv.exists() or sessions_csv.stat().st_size != member.file_size:
        if disk_usage(TEMP).free < member.file_size + 500 * 1024**2:
            raise RuntimeError('Недостаточно места для распаковки sessions.csv')
        with archive.open(member) as src, sessions_csv.open('wb') as dst:
            copyfileobj(src, dst, length=8 * 1024**2)
print('Файл действий подготовлен, ГБ:', round(sessions_csv.stat().st_size / 1024**3, 2))


Файл действий подготовлен, ГБ: 0.59


## Подготовка признаков действий

Таблица действий читается **один раз**, частями. Пользовательские признаки сохраняются во временной папке; при повторном запуске в той же среде готовый кеш используется без нового прохода по большому файлу.


In [3]:
cache = TEMP / 'user_features_2014.csv'
tracked = ['view_search_results', 'similar_listings', 'wishlist_content_update',
           'contact_host', 'message_post']
required = {'id', 'month', 'affiliate_channel', 'signup_app', 'signup_method',
            'signup_flow', 'booked', 'has_sessions', 'event_count'} | {
            f'did_{a}' for a in tracked}

if cache.exists():
    cached = pd.read_csv(cache)
    if required.issubset(cached.columns) and len(cached) == len(cohort) and set(cached['id']) == set(cohort['id']):
        features = cached
        print('Готовый кеш признаков загружен: повторное чтение действий не требуется.')
    else:
        raise ValueError('Кеш не соответствует выборке. Удалите временный кеш и запустите эту ячейку снова.')
else:
    target_ids = set(cohort['id'])
    event_counts = Counter()
    seen = {a: set() for a in tracked}
    total_rows = 0
    missing_user_id = 0
    session_user_ids = set()
    for chunk_number, chunk in enumerate(pd.read_csv(
        sessions_csv, usecols=['user_id', 'action_detail'], chunksize=500_000
    ), start=1):
        total_rows += len(chunk)
        missing_user_id += int(chunk['user_id'].isna().sum())
        session_user_ids.update(chunk['user_id'].dropna().unique())
        part = chunk[chunk['user_id'].isin(target_ids)]
        event_counts.update(part['user_id'].value_counts().to_dict())
        selected = part.loc[part['action_detail'].isin(tracked),
                            ['user_id', 'action_detail']].drop_duplicates()
        for action, group in selected.groupby('action_detail'):
            seen[action].update(group['user_id'])
        if chunk_number % 5 == 0:
            print('Обработано строк действий:', total_rows)
    print('Всего строк действий:', total_rows)
    print('Строк без user_id:', missing_user_id)
    print('Уникальных user_id в действиях:', len(session_user_ids))
    cohort['has_sessions'] = cohort['id'].isin(event_counts)
    features = cohort[['id', 'month', 'affiliate_channel', 'signup_app',
                       'signup_method', 'signup_flow', 'booked', 'has_sessions']].copy()
    features['event_count'] = features['id'].map(event_counts).fillna(0).astype(int)
    for action in tracked:
        features[f'did_{action}'] = features['id'].isin(seen[action])
    features.to_csv(cache, index=False)
    print('Кеш пользовательских признаков сохранён.')

assert features['id'].is_unique and len(features) == len(cohort)
assert (features['event_count'].gt(0) == features['has_sessions']).all()
print('Пользователей когорты с логами:', int(features['has_sessions'].sum()))


Обработано строк действий: 2500000
Обработано строк действий: 5000000
Обработано строк действий: 7500000
Обработано строк действий: 10000000
Всего строк действий: 10567737
Строк без user_id: 34496
Уникальных user_id в действиях: 135483
Кеш пользовательских признаков сохранён.
Пользователей когорты с логами: 73815


## Сводные таблицы

Для доли с бронированием по месяцам используются **все** пользователи выбранной когорты. При сравнении действий `content` и `direct` используются только Web-пользователи **с логами**. Знаменатели этих двух таблиц различаются.


In [5]:
by_month = features.groupby(['month', 'affiliate_channel'], dropna=False).agg(
    users=('id', 'size'), booked=('booked', 'sum'),
    with_sessions=('has_sessions', 'sum')
).reset_index()
by_month['booked_pct'] = (100 * by_month['booked'] / by_month['users']).round(2)
by_month.to_csv(OUT / 'channel_month_summary.csv', index=False)

web = features[features['signup_app'].eq('Web') & features['has_sessions'] &
               features['affiliate_channel'].isin(['content', 'direct'])]
rates = web.groupby('affiliate_channel').agg(
    users=('id', 'size'), booked=('booked', 'sum'),
    search_users=('did_view_search_results', 'sum'),
    contact_users=('did_contact_host', 'sum'),
    message_users=('did_message_post', 'sum'),
    wishlist_users=('did_wishlist_content_update', 'sum'),
    avg_events=('event_count', 'mean')
).reset_index()
for count_col, rate_col in [
    ('booked', 'booked_pct'), ('search_users', 'search_users_pct'),
    ('contact_users', 'contact_users_pct'), ('message_users', 'message_users_pct'),
    ('wishlist_users', 'wishlist_users_pct')
]:
    rates[rate_col] = (100 * rates[count_col] / rates['users']).round(2)
rates['avg_events'] = rates['avg_events'].round(2)
rates.to_csv(OUT / 'behavior_by_channel_web_rates.csv', index=False)
print('Строк в таблице «месяц × канал»:', len(by_month))
print(rates.to_string(index=False))
print('Созданы: channel_month_summary.csv и behavior_by_channel_web_rates.csv')


Строк в таблице «месяц × канал»: 46
affiliate_channel  users  booked  search_users  contact_users  message_users  wishlist_users  avg_events  booked_pct  search_users_pct  contact_users_pct  message_users_pct  wishlist_users_pct
          content   1630     181          1069            107            113            1062       43.79       11.10             65.58               6.56               6.93               65.15
           direct  33017   15004         18410          10867          14885           23787       71.41       45.44             55.76              32.91              45.08               72.04
Созданы: channel_month_summary.csv и behavior_by_channel_web_rates.csv


## Ограничения результата

Данные наблюдательные: различия между каналами не доказывают влияние канала или интерфейса. В `sessions.csv` нет времени каждого действия: оно могло произойти после бронирования. В этих таблицах нет стоимости бронирования и экспериментальных групп. Построчный файл признаков хранится только во временной папке, агрегаты — в выходной папке Kaggle.


## Исследовательская проверка: Web и basic

Сравниваются доли пользователей с указанной `date_first_booking` среди зарегистрировавшихся с января по июнь 2014 года через `Web`, со способом регистрации `basic` и каналом `content` или `direct`.

**H₀:** доли в двух каналах равны.  
**H₁:** доли различаются.

Используется двусторонний z-тест двух пропорций при уровне значимости 0,05. Для разницы `direct − content` рассчитывается 95% интервал методом Ньюкомба. Подгруппа `basic` выбрана после предварительного изучения данных, поэтому проверка носит исследовательский характер.


In [1]:
from pathlib import Path
import pandas as pd

path = next(Path("/kaggle/input").rglob("train_users_2.csv.zip"))

users = pd.read_csv(
    path,
    compression="zip",
    usecols=[
        "date_account_created",
        "date_first_booking",
        "signup_app",
        "signup_method",
        "affiliate_channel",
    ],
)

registration = pd.to_datetime(users["date_account_created"], errors="coerce")

cohort = users.loc[
    registration.between("2014-01-01", "2014-06-30")
    & users["signup_app"].eq("Web")
    & users["signup_method"].eq("basic")
    & users["affiliate_channel"].isin(["content", "direct"])
].copy()

cohort["has_booking"] = pd.to_datetime(
    cohort["date_first_booking"], errors="coerce"
).notna()

result = (
    cohort.groupby("affiliate_channel", as_index=False)
    .agg(
        users=("has_booking", "size"),
        booked=("has_booking", "sum"),
    )
)

result["share_pct"] = (100 * result["booked"] / result["users"]).round(2)
display(result)

,affiliate_channel,users,booked,share_pct
0,content,940,129,13.72
1,direct,25821,11822,45.78


In [2]:
from statsmodels.stats.proportion import (
    proportions_ztest,
    confint_proportions_2indep,
)

groups = result.set_index("affiliate_channel")

direct = groups.loc["direct"]
content = groups.loc["content"]

x_direct, n_direct = int(direct["booked"]), int(direct["users"])
x_content, n_content = int(content["booked"]), int(content["users"])

z_stat, p_value = proportions_ztest(
    [x_direct, x_content],
    [n_direct, n_content],
    alternative="two-sided",
)

ci_low, ci_high = confint_proportions_2indep(
    x_direct, n_direct,
    x_content, n_content,
    compare="diff",
    method="newcomb",
    alpha=0.05,
)

difference_pp = 100 * (
    x_direct / n_direct - x_content / n_content
)

print(f"Разница direct − content: {difference_pp:.2f} п. п.")
print(f"z-статистика: {z_stat:.2f}")
print(f"p-value: {p_value:.3e}")
print(f"95% интервал разницы: [{100*ci_low:.2f}; {100*ci_high:.2f}] п. п.")

Разница direct − content: 32.06 п. п.
z-статистика: 19.42
p-value: 5.005e-84
95% интервал разницы: [29.64; 34.20] п. п.


In [3]:
signup_date = pd.to_datetime(
    cohort["date_account_created"], errors="coerce"
)
booking_date = pd.to_datetime(
    cohort["date_first_booking"], errors="coerce"
)

cohort["booking_before_signup"] = (
    booking_date.notna() & (booking_date < signup_date)
)

print("Всего таких записей:", cohort["booking_before_signup"].sum())
print(
    cohort.groupby("affiliate_channel")["booking_before_signup"]
    .sum()
)

Всего таких записей: 0
affiliate_channel
content    0
direct     0
Name: booking_before_signup, dtype: int64


In [4]:
cohort["month"] = pd.to_datetime(
    cohort["date_account_created"]
).dt.to_period("M").astype(str)

monthly = (
    cohort.groupby(["month", "affiliate_channel"], as_index=False)
    .agg(
        users=("has_booking", "size"),
        booked=("has_booking", "sum"),
    )
)

monthly["share_pct"] = (
    100 * monthly["booked"] / monthly["users"]
).round(2)

display(monthly)

,month,affiliate_channel,users,booked,share_pct
0,2014-01,content,237,20,8.44
1,2014-01,direct,4013,1781,44.38
2,2014-02,content,195,29,14.87
3,2014-02,direct,3669,1774,48.35
4,2014-03,content,144,19,13.19
5,2014-03,direct,4098,1895,46.24
6,2014-04,content,139,25,17.99
7,2014-04,direct,4074,1847,45.34
8,2014-05,content,151,24,15.89
9,2014-05,direct,4794,2221,46.33


### Интерпретация результата

В подгруппе Web-пользователей со способом регистрации `basic`
доля с зафиксированным первым бронированием составила 13,72%
для `content` (129 из 940) и 45,78% для `direct`
(11 822 из 25 821). Разница `direct − content` — 32,06 п. п.

Двусторонний тест двух пропорций: z = 19,42;
p ≈ 5,005 × 10⁻⁸⁴. Расчётный 95% интервал для разницы —
29,64–34,20 п. п. При допущениях теста гипотеза о равенстве
долей отвергается.

В выбранной подгруппе нет записей, где дата первого бронирования
раньше регистрации. Во всех шести месяцах регистрации доля у
`direct` выше, чем у `content`; отдельные месячные тесты
не проводились.

Это исследовательская проверка наблюдательных данных.
Подгруппа выбрана после знакомства с данными, пользователи
не распределялись по каналам случайно, а для бронирования
не задано единое окно после регистрации. Результат показывает
различие долей, но не доказывает влияние канала.